In [21]:
import sys
from pathlib import Path
import dill
import pandas as pd

# get rid of warning
import warnings
warnings.filterwarnings("ignore")

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "src" / "config.py").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DIR, PROCESSED_DIR, SRC_DIR
from src.extraction.extract import (
    backfill_events_json,
    build_season_events,
    event_coverage_report,
    extract_all_events,
    get_epl_matches,
    get_matches_formatted,
    normalize_existing_parts,
)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
# dill.load_session('../saves/ex.db')

In [4]:
from statsbombpy import sb
from tqdm import tqdm

comps = sb.competitions()

In [11]:
results = []

for _, row in tqdm(comps.iterrows(), total=len(comps)):
    comp_id = row["competition_id"]
    season_id = row["season_id"]
    
    try:
        matches = sb.matches(competition_id=comp_id, season_id=season_id)
        match_count = len(matches)

        event_count = 0

        # sample a few matches to estimate event density (faster than full scan)
        sample_matches_id = matches["match_id"].head(min(5, len(matches)))

        for m_id in sample_matches_id:
            events = sb.events(match_id=m_id)
            event_count += len(events)

        avg_events_per_match = event_count / len(sample_matches_id) if len(sample_matches_id) > 0 else 0
        est_total_events = avg_events_per_match * match_count

        results.append({
            "competition": row["competition_name"],
            "country": row["country_name"],
            "season": season_id,
            "season_name": row["season_name"],
            "matches": match_count,
            "avg_events_per_match": avg_events_per_match,
            "estimated_total_events": est_total_events
        })

    except Exception as e:
        print(f"Skipped {comp_id}-{season_id}: {e}")

df = pd.DataFrame(results)

100%|██████████| 80/80 [00:26<00:00,  3.03it/s]


### Survey note

`estimated_total_events` is a **projection only**: it samples **5 matches** per competition-season, computes `avg_events_per_match`, then multiplies by total match count. It does **not** download or save events. Actual volume comes from the extraction cells below.

In [12]:
df.sort_values("estimated_total_events", ascending=False).head(10)

,competition,country,season,season_name,matches,avg_events_per_match,estimated_total_events
63,Ligue 1,France,27,2015/2016,377,3786.6,1427548.2
68,Premier League,England,27,2015/2016,380,3649.0,1386620.0
70,Serie A,Italy,27,2015/2016,380,3491.0,1326580.0
45,La Liga,Spain,27,2015/2016,380,3179.0,1208020.0
58,Liga F,Spain,281,2023/2024,240,3672.4,881376.0
25,FA Women's Super League,England,281,2023/2024,132,3655.0,482460.0
38,Frauen Bundesliga,Germany,281,2023/2024,132,3534.4,466540.8
66,NWSL,United States of America,107,2023,137,3319.8,454812.6
26,FA Women's Super League,England,90,2020/2021,131,3431.0,449461.0
72,Serie A Women,Italy,281,2023/2024,130,3369.0,437970.0


## Premier League extraction

Goal: analyze playstyle change (2003/04 vs 2015/16). The survey above ranked competitions; this section downloads **all** EPL match events via `src.extract`.

In [13]:
# 2003/2004 season
matches_2004 = get_matches_formatted(2,  44)
df_2004 = pd.DataFrame(matches_2004)
df_2004.to_csv(PROCESSED_DIR/"matches_2004.csv", index=False)

# 2015/2016 season
matches_2016 = get_matches_formatted(2, 27)
df_2016 = pd.DataFrame(matches_2016)
df_2016.to_csv(PROCESSED_DIR/"matches_2016.csv", index=False)


In [14]:
# resume=True skips match_ids already saved under data/raw/events_parts/
events_df_2004 = extract_all_events(df_2004, RAW_DIR, resume=True)
print(f"Extracted {len(events_df_2004):,} events from {events_df_2004['match_id'].nunique()} matches")

events_df_2016 = extract_all_events(df_2016, RAW_DIR, resume=True)
print(f"Extracted {len(events_df_2016):,} events from {events_df_2016['match_id'].nunique()} matches")

# Doc-aligned processed parquets (normalized columns + spatial splits)
build_season_events(df_2004, "2004", RAW_DIR, PROCESSED_DIR)
build_season_events(df_2016, "2016", RAW_DIR, PROCESSED_DIR)

Resuming: skipping 418 matches already on disk, 0 remaining.


Extracting events: 0it [00:00, ?it/s]


Extracted 129,401 events from 38 matches
Resuming: skipping 418 matches already on disk, 0 remaining.


Extracting events: 0it [00:00, ?it/s]


Extracted 1,313,773 events from 380 matches
Saved 129,401 events to /Users/sothea/Documents/Files/Code/2009-Barcelona-analysis/notebooks/data/processed/events_2004.parquet
Saved 1,313,773 events to /Users/sothea/Documents/Files/Code/2009-Barcelona-analysis/notebooks/data/processed/events_2016.parquet


,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,shot_saved_off_target,shot_redirect,goalkeeper_shot_saved_to_post,shot_saved_to_post,player_off_permanent,goalkeeper_lost_out,half_start_late_video_start,goalkeeper_lost_in_play,goalkeeper_penalty_saved_to_post,goalkeeper_saved_to_post
0,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313768,NaN,None,None,None,"[75.6, 31.4]",None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313769,NaN,None,None,None,None,None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313770,NaN,None,None,None,None,None,Head,True,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313771,NaN,None,None,None,None,None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
sample = pd.read_parquet(PROCESSED_DIR / "events_2004.parquet")
print(f"Normalized columns ", sample.columns.tolist())
# print(sample[["type", "location_x", "location_y", "pass_complete", "pass_end_x"]].head())

Normalized columns  ['ball_receipt_outcome', 'ball_recovery_recovery_failure', 'block_deflection', 'block_offensive', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'dribble_overrun', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_offensive', 'foul_committed_penalty', 'foul_committed_type', 'foul_won_advantage', 'foul_won_defensive', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_punched_out', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_inswinging', 'pas

In [16]:
matches_cov = pd.concat([
    df_2004.assign(season="2003/2004"),
    df_2016.assign(season="2015/2016"),
])
events_cov = pd.concat([
    pd.read_parquet(PROCESSED_DIR / "events_2004.parquet", columns=["match_id"]),
    pd.read_parquet(PROCESSED_DIR / "events_2016.parquet", columns=["match_id"]),
])

coverage = event_coverage_report(matches_cov, events_cov)
print(f"Total events saved: {len(events_cov):,}\n")
print("Matches vs events captured by season:")
print(coverage)

# Compare 2015/16 to survey projection (season_id 27 in survey df)
epl_survey = df[(df["competition"] == "Premier League") & (df["season"] == 27)]
if not epl_survey.empty:
    est = epl_survey["estimated_total_events"].iloc[0]
    actual = coverage.loc["2015/2016", "events"] if "2015/2016" in coverage.index else 0
    print(f"\n2015/16 survey estimate: {est:,.0f} | actual events: {actual:,}")

Total events saved: 1,443,174

Matches vs events captured by season:
           matches  matches_with_events   events  missing_matches
season                                                           
2003/2004       38                   38   129401                0
2015/2016      380                  380  1313773                0

2015/16 survey estimate: 1,386,620 | actual events: 1,313,773


In [17]:
# One-time migration for data extracted before the doc-aligned pipeline.
# This normalizes existing parquet parts without re-downloading events, then backfills doc-faithful JSON.
all_match_ids = pd.concat([df_2004["match_id"], df_2016["match_id"]]).tolist()
normalize_existing_parts(all_match_ids, RAW_DIR)
backfill_events_json(all_match_ids, RAW_DIR, resume=True)
build_season_events(df_2004, "2004", RAW_DIR, PROCESSED_DIR)
build_season_events(df_2016, "2016", RAW_DIR, PROCESSED_DIR)

Normalizing parts: 100%|██████████| 418/418 [00:01<00:00, 215.72it/s]


JSON backfill: skipping 418 matches, 0 remaining.


Backfilling JSON: 0it [00:00, ?it/s]


Saved 129,401 events to /Users/sothea/Documents/Files/Code/2009-Barcelona-analysis/notebooks/data/processed/events_2004.parquet
Saved 1,313,773 events to /Users/sothea/Documents/Files/Code/2009-Barcelona-analysis/notebooks/data/processed/events_2016.parquet


,bad_behaviour_card,ball_receipt_outcome,ball_recovery_recovery_failure,block_deflection,carry_end_location,clearance_aerial_won,clearance_body_part,clearance_head,clearance_left_foot,clearance_other,...,shot_saved_off_target,shot_redirect,goalkeeper_shot_saved_to_post,shot_saved_to_post,player_off_permanent,goalkeeper_lost_out,half_start_late_video_start,goalkeeper_lost_in_play,goalkeeper_penalty_saved_to_post,goalkeeper_saved_to_post
0,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,None,None,None,None,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1313768,NaN,None,None,None,"[75.6, 31.4]",None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313769,NaN,None,None,None,None,None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313770,NaN,None,None,None,None,None,Head,True,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1313771,NaN,None,None,None,None,None,None,None,None,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
dill.dump_session('../saves/extraction.db')